# Code_2 — Reference-Based Metrics (baselines that need the answer key)

**GPU: yes** (short — NLI and BERTScore). CPU works but BERTScore is slow.

Scores the single-error benchmark with the usual reference-based metrics. **Every metric here
compares the generated call with the correct call**, which is not available at run time — they are
reported as *detection-only, non-deployable* baselines.

| Metric | What is compared |
|---|---|
| Exact Match (normalised) | normalised call == normalised reference |
| Exact Match (raw) | raw call == raw reference (envelope included) |
| Token F1 | token overlap of the normalised JSON strings |
| Function Match / Param-Set Match | function name / parameter-name set only |
| NLI | P(entailment) between natural-language renderings, DeBERTa cross-encoder |
| BERTScore | F1 of RoBERTa-large embeddings of the normalised JSON strings |

For every metric the decision threshold is chosen on the **tune** part (maximising balanced
accuracy) and applied unchanged to the **test** part, for both the random and the
function-disjoint split. Output: `runs/<run_id>/reference_metrics.json`.

In [ ]:
import os, sys, json, importlib, subprocess, time
from collections import Counter, defaultdict

IN_COLAB = os.path.exists('/content')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/hallucination_project'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bert_score'], check=True)
else:
    DRIVE_ROOT = os.environ.get('TOOLHALL_ROOT', os.path.abspath('hallucination_project'))
CODE_DIR = os.path.join(DRIVE_ROOT, 'code')
sys.path.insert(0, CODE_DIR)
import toolhall_core as th
importlib.reload(th)

NOTEBOOK = 'Code_2'
CFG = th.load_config(os.path.join(CODE_DIR, 'config.json'))
CFG['drive_root'] = DRIVE_ROOT
P = th.Paths(CFG)
th.set_seed(CFG['seed'])
NB, ALPHA = CFG['bootstrap']['n'], CFG['bootstrap']['alpha']

single = th.load_json(P.data_file('benchmark_single.json'), CFG, kind='data')
splits = th.load_json(P.data_file('splits.json'), CFG, kind='data')
print(f"run_id={CFG['run_id']} | benchmark: {len(single)} | "
      f"tune/test random {len(splits['random']['tune'])}/{len(splits['random']['test'])} | "
      f"function-disjoint {len(splits['function_disjoint']['tune'])}/{len(splits['function_disjoint']['test'])}")

## Surface metrics

In [ ]:
def norm_pair(ex):
    return th.normalize_tool_call(ex['ground_truth_tool_call']), th.normalize_tool_call(ex['generated_tool_call'])


def m_exact_norm(ex):
    g, n = norm_pair(ex)
    return 1.0 if g and n and th.canonical_json(g) == th.canonical_json(n) else 0.0


def m_exact_raw(ex):
    g = th.parse_raw_call(ex['ground_truth_tool_call'])
    n = th.parse_raw_call(ex['generated_tool_call'])
    return 1.0 if g is not None and n is not None and th.canonical_json(g) == th.canonical_json(n) else 0.0


def m_token_f1(ex):
    g, n = norm_pair(ex)
    if not g or not n:
        return 0.0
    gt, nt = set(json.dumps(g).lower().split()), set(json.dumps(n).lower().split())
    inter = gt & nt
    if not inter:
        return 0.0
    p, r = len(inter) / len(nt), len(inter) / len(gt)
    return 2 * p * r / (p + r)


def m_function(ex):
    g, n = norm_pair(ex)
    return 1.0 if g and n and g['function'] == n['function'] else 0.0


def m_param_set(ex):
    g, n = norm_pair(ex)
    return 1.0 if g and n and set(g['parameters']) == set(n['parameters']) else 0.0


SURFACE = {'exact_match_normalised': m_exact_norm, 'exact_match_raw': m_exact_raw,
           'token_f1': m_token_f1, 'function_match': m_function, 'param_set_match': m_param_set}
scores = {name: [fn(ex) for ex in single] for name, fn in SURFACE.items()}
print({k: round(sum(v) / len(v), 4) for k, v in scores.items()}, '(mean similarity)')

## NLI (DeBERTa cross-encoder) — label order read from the model config

The previous version took column 0 minus column 2 as "entailment minus contradiction", but for
`cross-encoder/nli-deberta-v3-small` the columns are *contradiction, entailment, neutral*, so it
was actually contradiction minus neutral. Here the entailment column is looked up by name.

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
NLI_NAME = CFG['models']['nli_model']
nli_tok = AutoTokenizer.from_pretrained(NLI_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_NAME).to(DEVICE).eval()
id2label = {int(k): v.lower() for k, v in nli_model.config.id2label.items()}
ENT = next(i for i, l in id2label.items() if 'entail' in l)
print('NLI labels:', id2label, '-> entailment column', ENT, '| device', DEVICE)


def to_text(norm):
    if not norm:
        return ''
    ps = norm.get('parameters') or {}
    if not ps:
        return f"call function {norm['function']} with no parameters"
    return f"call function {norm['function']} with parameters " + ', '.join(f"{k} is {v}" for k, v in ps.items())


def nli_entail(pairs, bs=32):
    out = []
    for i in range(0, len(pairs), bs):
        b = pairs[i:i + bs]
        enc = nli_tok([p for p, _ in b], [h for _, h in b], return_tensors='pt', truncation=True,
                      max_length=512, padding=True).to(DEVICE)
        with torch.no_grad():
            probs = torch.softmax(nli_model(**enc).logits, dim=-1).cpu().numpy()
        out += [float(r[ENT]) for r in probs]
    return out


t0 = time.time()
pairs_norm = [(to_text(g), to_text(n)) for g, n in (norm_pair(ex) for ex in single)]
pairs_raw = [(json.dumps(th.parse_raw_call(ex['ground_truth_tool_call']) or {}),
              json.dumps(th.parse_raw_call(ex['generated_tool_call']) or {})) for ex in single]
scores['nli_normalised'] = nli_entail(pairs_norm)
scores['nli_raw_json'] = nli_entail(pairs_raw)
print(f"NLI done in {time.time() - t0:.0f}s")
del nli_model
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

## BERTScore (RoBERTa-large)

In [ ]:
from bert_score import score as bert_score_fn
t0 = time.time()
gen_s = [th.canonical_json(n) if n else 'empty' for _, n in (norm_pair(ex) for ex in single)]
ref_s = [th.canonical_json(g) if g else 'empty' for g, _ in (norm_pair(ex) for ex in single)]
_, _, F1 = bert_score_fn(gen_s, ref_s, model_type=CFG['models']['bertscore_model'], lang='en',
                         verbose=False, device=DEVICE, rescale_with_baseline=False, batch_size=32)
scores['bertscore_f1'] = [float(x) for x in F1]
print(f"BERTScore done in {time.time() - t0:.0f}s")

## Thresholds on tune, results on test (both splits)

In [ ]:
y_all = [int(ex['is_hallucinated']) for ex in single]
labels_all = [ex['hallucination_type'] for ex in single]


def fit_threshold(sc, idx):
    """Flag as hallucinated when similarity < t; t maximises balanced accuracy on `idx`."""
    cands = sorted(set(round(sc[i], 6) for i in idx))
    cands = [c + 1e-9 for c in cands] + [min(cands) - 1e-9] if cands else [0.5]
    best_t, best = cands[0], -1
    yt = [y_all[i] for i in idx]
    for t in cands:
        b = th.binary_metrics(yt, [int(sc[i] < t) for i in idx])['balanced_accuracy']
        if b > best:
            best, best_t = b, t
    return best_t, best


results, per_example = {}, {}
for split_name in ['random', 'function_disjoint']:
    tune, test = splits[split_name]['tune'], splits[split_name]['test']
    results[split_name] = {}
    for name, sc in scores.items():
        t, tune_bacc = fit_threshold(sc, tune)
        yt = [y_all[i] for i in test]
        yp = [int(sc[i] < t) for i in test]
        rep = {'threshold': t, 'tune_balanced_accuracy': tune_bacc,
               'binary': th.binary_metrics(yt, yp),
               'ci': {m: th.bootstrap_ci(lambda a, b, m=m: th.binary_metrics(a, b)[m], yt, yp,
                                         n=NB, alpha=ALPHA, seed=CFG['seed'])
                      for m in ('accuracy', 'f1', 'balanced_accuracy')},
               'flag_rate_by_label': {}}
        for lab in th.ALL_LABELS:
            ii = [k for k, i in enumerate(test) if labels_all[i] == lab]
            if ii:
                rep['flag_rate_by_label'][lab] = float(np.mean([yp[k] for k in ii]))
        results[split_name][name] = rep
        per_example.setdefault(split_name, {})[name] = {single[i]['id']: int(sc[i] < t) for i in test}

for split_name in results:
    th.print_header(f"REFERENCE-BASED DETECTION — {split_name} test split "
                    f"(n={len(splits[split_name]['test'])})")
    print(f"{'metric':<24}{'acc':>7}{'bal.acc':>9}{'prec':>7}{'rec':>7}{'F1':>7}{'FPR':>7}{'FNR':>7}   balanced acc 95% CI")
    for name, rep in results[split_name].items():
        b = rep['binary']
        print(f"{name:<24}{b['accuracy']:>7.3f}{b['balanced_accuracy']:>9.3f}{b['precision']:>7.3f}"
              f"{b['recall']:>7.3f}{b['f1']:>7.3f}{b['fpr']:>7.3f}{b['fnr']:>7.3f}   "
              f"{th.fmt_ci(rep['ci']['balanced_accuracy'])}")
    print(f"\nflag rate by true label (NONE column = false-positive rate):")
    print(f"{'metric':<24}" + ''.join(f"{l[:9]:>10}" for l in th.ALL_LABELS))
    for name, rep in results[split_name].items():
        print(f"{name:<24}" + ''.join(f"{rep['flag_rate_by_label'].get(l, float('nan')):>10.3f}"
                                      for l in th.ALL_LABELS))

## Save

In [ ]:
out = {'note': 'All metrics in this file REQUIRE the reference call; they are not deployable detectors.',
       'threshold_rule': 'flag if similarity < t; t maximises balanced accuracy on the tune part',
       'nli_model': NLI_NAME, 'nli_labels': id2label, 'bertscore_model': CFG['models']['bertscore_model'],
       'results': results, 'per_example_flags': per_example,
       'hardware': th.hardware_info(), 'packages': th.package_versions()}
th.save_json(out, P.run_file('reference_metrics.json'), CFG, NOTEBOOK)
print('saved ->', P.run_file('reference_metrics.json'))
print('Code_2 complete.')